In [ ]:
import socket
import struct

A = 1.4
B = 0.3
x, y = 0.0, 0.0
HEADER = struct.Struct(">I")


def henon_map(x, y):
    return 1 - A * x**2 + y, B * x


def generate_chaos_sequence(length):
    """Advance the shared map state and return `length` keystream bytes."""
    global x, y
    seq = []
    for _ in range(length):
        x, y = henon_map(x, y)
        seq.append(int(x * 255) % 256)
    return seq


def decrypt_message(encrypted_bytes):
    keystream = generate_chaos_sequence(len(encrypted_bytes))
    plaintext = bytes(e ^ k for e, k in zip(encrypted_bytes, keystream))
    return plaintext.decode("utf-8")


def recv_exact(conn, size):
    """Read exactly `size` bytes, or return None if the peer closed first."""
    buf = b""
    while len(buf) < size:
        chunk = conn.recv(size - len(buf))
        if not chunk:
            return None
        buf += chunk
    return buf

In [ ]:
HOST = "0.0.0.0"
PORT = 65432

server_socket = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
server_socket.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
server_socket.bind((HOST, PORT))
server_socket.listen(1)

print(f"Server listening on port {PORT}...")

conn, addr = server_socket.accept()
print(f"Connected by {addr}")

while True:
    header = recv_exact(conn, HEADER.size)
    if header is None:
        break
    (length,) = HEADER.unpack(header)
    data = recv_exact(conn, length)
    if data is None:
        break
    print(f"Received (encrypted): {list(data)}")
    print(f"Received (decrypted): {decrypt_message(data)}")

conn.close()
server_socket.close()